The Scenario: You are given three separate datasets: "Customer Profiles" (age, region, subscription tier), "Product Catalog" (category, weight, unit price), and "Order Transactions" (who bought what, when, and delivery times).

In [2]:
import pandas as pd
customer = pd.read_csv("customers.csv")
product = pd.read_csv("products.csv")
order1 = pd.read_csv("orders_jan.csv")
order2 = pd.read_csv("orders_feb.csv")

Combining Datasets: You must use pd.concat() to stitch together the order transactions from "January" and "February". Then, use pd.merge() to join the combined 'Order Transactions' table with the 'Product Catalog' table so you know the category and price of the item purchased, acting like a SQL left join.

In [10]:
combined = pd.concat([order1, order2], ignore_index=True)
combined
merged = pd.merge(combined, product, on='Product_ID', how='inner')
merged

,Transaction_ID,cust_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Warehouse_Route_ID
0,T00001,C0546,P0146,4,2024-01-26,2,WH-A1
1,T00002,C0609,P0938,1,2024-01-22,6,WH-C4
2,T00003,C0169,P0057,3,2024-01-07,6,WH-B3
3,T00004,C0681,P0723,1,2024-01-10,1,WH-A9
4,T00005,C0663,P0780,3,2024-01-28,1,WH-A7
...,...,...,...,...,...,...,...
1995,T01996,C0637,P0623,3,2024-02-19,3,WH-C2
1996,T01997,C0716,P0210,3,2024-02-09,4,WH-A8
1997,T01998,C0573,P0513,1,2024-02-09,1,WH-B4
1998,T01999,C0762,P0058,3,2024-02-13,2,WH-C9


Modifying DataFrames: You create a new calculated column called Total_Revenue by multiplying the Quantity column by the Unit_Price column. Use .drop() to remove redundant warehouse routing IDs, and use .rename() to fix inconsistently capitalized columnbn s (e.g., changing cust_ID to Customer_ID).

In [9]:
merged['Total_Revenue'] = (
    merged['Quantity'] * merged['Unit_Price']
)
merged
clear = merged.drop(columns=["Warehouse_Route_ID"], errors="ignore")
clear
combined.rename(columns={"cust_ID": "Customer_ID"}, inplace=True)
combined

,Transaction_ID,cust_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Warehouse_Route_ID,Product_Category,Weight_kg,Unit_Price,Total_Revenue
0,T00001,C0546,P0146,4,2024-01-26,2,WH-A1,Home & Garden,11.97,403.69,1614.76
1,T00002,C0609,P0938,1,2024-01-22,6,WH-C4,Toys,11.11,152.34,152.34
2,T00003,C0169,P0057,3,2024-01-07,6,WH-B3,Clothing,10.97,8.16,24.48
3,T00004,C0681,P0723,1,2024-01-10,1,WH-A9,Electronics,11.15,488.40,488.40
4,T00005,C0663,P0780,3,2024-01-28,1,WH-A7,Sports,7.34,476.43,1429.29
...,...,...,...,...,...,...,...,...,...,...,...
1995,T01996,C0637,P0623,3,2024-02-19,3,WH-C2,Toys,7.38,298.92,896.76
1996,T01997,C0716,P0210,3,2024-02-09,4,WH-A8,Toys,4.08,55.91,167.73
1997,T01998,C0573,P0513,1,2024-02-09,1,WH-B4,Toys,2.94,85.87,85.87
1998,T01999,C0762,P0058,3,2024-02-13,2,WH-C9,Home & Garden,14.51,489.29,1467.87


Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the Delivery_Days column into delivery performance buckets: "Early" (1-2 days), "On-Time" (3-4 days), or "Delayed" (5+ days).

In [12]:
combined['Delivery_Performance'] = combined['Delivery_Days'].apply(lambda x: "Early" if x <= 2 else ("On-Time" if x <= 4 else "Delayed"))
combined

,Transaction_ID,cust_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Warehouse_Route_ID,Delivery_Performance
0,T00001,C0546,P0146,4,2024-01-26,2,WH-A1,Early
1,T00002,C0609,P0938,1,2024-01-22,6,WH-C4,Delayed
2,T00003,C0169,P0057,3,2024-01-07,6,WH-B3,Delayed
3,T00004,C0681,P0723,1,2024-01-10,1,WH-A9,Early
4,T00005,C0663,P0780,3,2024-01-28,1,WH-A7,Early
...,...,...,...,...,...,...,...,...
1995,T01996,C0637,P0623,3,2024-02-19,3,WH-C2,On-Time
1996,T01997,C0716,P0210,3,2024-02-09,4,WH-A8,On-Time
1997,T01998,C0573,P0513,1,2024-02-09,1,WH-B4,Early
1998,T01999,C0762,P0058,3,2024-02-13,2,WH-C9,Early


Grouping and Aggregation: Using .groupby(), you group the data by Product_Category and use .agg() to find the total revenue generated, the average delivery days, and the unique count of Customer_IDs who purchased from that category.

In [ ]:
summary = orders_merged.groupby('Product_Category').agg({
    'Total_Revenue': 'sum',
    'Delivery_Days': 'mean',
    'Customer_ID': 'nunique'
})

print(summary)

Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a business-reporting matrix showing "Customer Region" as rows, "Delivery Performance" (Early/On-Time/Delayed) as columns, and the "Total Revenue" as the values.